# Finetune InternImage (Faster R-CNN) cho Ship Detection trên Sentinel-2

**Bối cảnh:** ~2000 ảnh Sentinel-2, 800×800px, tàu kích thước 4–18px, nhãn bbox-only dạng COCO JSON.

**Kiến trúc:** Repo `OpenGVLab/InternImage` chỉ release config Mask R-CNN / Cascade Mask R-CNN, không có Faster R-CNN thuần. Vì Mask R-CNN = Faster R-CNN + mask head, notebook này lấy config Mask R-CNN chính thức (`mask_rcnn_internimage_t_fpn_1x_coco.py`) làm `_base_`, sau đó **xóa `mask_head`/`mask_roi_extractor`** để có đúng kiến trúc Faster R-CNN, tận dụng toàn bộ pretrain COCO của backbone/FPN/RPN/bbox_head.

**Backbone:** InternImage-T (30M params) — phù hợp với dataset ~2000 ảnh, tránh overfit so với S/B/L/XL.

**Các điều chỉnh cho object nhỏ (4–18px):**
- Giảm `anchor_generator.scales` (mặc định cho anchor nhỏ nhất 32px, quá to với tàu 4–18px)
- Giảm ngưỡng IoU của assigner ở RPN/RCNN để anchor nhỏ dễ match với GT nhỏ
- Bật `fp16` + `with_cp` (gradient checkpointing) để tiết kiệm VRAM trên GPU Kaggle (T4/P100, thường 16GB)

**Logging:** dùng `TextLoggerHook` mặc định của MMDetection (loss theo từng iteration) + 1 custom hook ghi tóm tắt theo epoch ra CSV (loss trung bình, lr, thời gian/epoch, VRAM), cộng với `EvalHook` đánh giá COCO mAP (bao gồm AP_small) sau mỗi epoch.

---
⚠️ **Trước khi chạy:** sửa các đường dẫn trong cell `CFG` ở Section 0 cho khớp với dataset của anh (dataset đã được chia sẵn train/val/test — mỗi split có thư mục ảnh + file COCO JSON riêng, notebook **không** chia lại).

## Section 0 — Cấu hình chung (SỬA CHỖ NÀY TRƯỚC KHI CHẠY)

In [1]:
import os

CFG = dict(
    # ---- Dataset đầu vào (đã được chia sẵn train/val/test, KHÔNG chia lại) ----
    # Mỗi split có thư mục ảnh + file COCO JSON riêng. Nếu ảnh của 3 split nằm chung
    # 1 thư mục thì chỉ cần trỏ cả 3 *_IMAGES_DIR về cùng đường dẫn đó.
    TRAIN_IMAGES_DIR = "/kaggle/input/datasets/hunglq/annotated-sentinel/train",
    TRAIN_JSON       = "/kaggle/input/datasets/hunglq/annotated-sentinel/train/_annotations.coco.json",
    VAL_IMAGES_DIR   = "/kaggle/input/datasets/hunglq/annotated-sentinel/valid",
    VAL_JSON         = "/kaggle/input/datasets/hunglq/annotated-sentinel/valid/_annotations.coco.json",
    TEST_IMAGES_DIR  = "/kaggle/input/datasets/hunglq/annotated-sentinel/test",
    TEST_JSON        = "/kaggle/input/datasets/hunglq/annotated-sentinel/test/_annotations.coco.json",
    CLASSES      = ("ship",),   # sửa nếu có nhiều hơn 1 lớp

    # ---- Backbone / method ----
    BACKBONE_SIZE = "t",   # t | s | b | l | xl  (khuyến nghị "t" cho ~2000 ảnh)
    SCHEDULE      = "1x",  # 1x | 3x (dùng để tìm đúng checkpoint pretrain trên HuggingFace)

    # ---- Working dirs (đều nằm trong /kaggle/working để giữ được sau khi notebook chạy xong) ----
    WORK_ROOT     = "/kaggle/working",
    REPO_DIR      = "/kaggle/working/InternImage",
    DATA_DIR      = "/kaggle/working/data",
    WORK_DIR      = "/kaggle/working/work_dirs/finetune_ship",
    PRETRAIN_DIR  = "/kaggle/working/pretrained",

    # ---- Training hyperparams ----
    IMG_SIZE        = 800,
    BATCH_SIZE      = 2,      # per GPU; giảm xuống 1 nếu OOM
    WORKERS         = 2,
    LR              = 2e-5,   # nhỏ hơn nhiều so với train from scratch (1e-4) vì đang finetune
    MAX_EPOCHS      = 12,
    WARMUP_ITERS    = 200,
    LR_STEP         = [8, 11],
    LOG_INTERVAL    = 20,     # log loss mỗi 20 iteration
    EVAL_INTERVAL   = 1,      # đánh giá mAP mỗi 1 epoch
    CHECKPOINT_MAX_KEEP = 3,
    USE_FP16        = True,
    SEED            = 42,
)



os.makedirs(CFG["WORK_ROOT"], exist_ok=True)
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
os.makedirs(CFG["WORK_DIR"], exist_ok=True)
os.makedirs(CFG["PRETRAIN_DIR"], exist_ok=True)

for split in ("TRAIN", "VAL", "TEST"):
    img_dir, ann_json = CFG[f"{split}_IMAGES_DIR"], CFG[f"{split}_JSON"]
    assert os.path.exists(img_dir), f"Không tìm thấy thư mục ảnh ({split}): {img_dir}"
    assert os.path.exists(ann_json), f"Không tìm thấy file COCO JSON ({split}): {ann_json}"

print("CFG OK.")
print("  Train:", CFG["TRAIN_IMAGES_DIR"], "|", CFG["TRAIN_JSON"])
print("  Val  :", CFG["VAL_IMAGES_DIR"], "|", CFG["VAL_JSON"])
print("  Test :", CFG["TEST_IMAGES_DIR"], "|", CFG["TEST_JSON"])


CFG OK.
  Train: /kaggle/input/datasets/hunglq/annotated-sentinel/train | /kaggle/input/datasets/hunglq/annotated-sentinel/train/_annotations.coco.json
  Val  : /kaggle/input/datasets/hunglq/annotated-sentinel/valid | /kaggle/input/datasets/hunglq/annotated-sentinel/valid/_annotations.coco.json
  Test : /kaggle/input/datasets/hunglq/annotated-sentinel/test | /kaggle/input/datasets/hunglq/annotated-sentinel/test/_annotations.coco.json


## Section 1 — Kiểm tra GPU & môi trường

Bước này chỉ để chẩn đoán sớm — nếu không thấy GPU hoặc `nvcc` không khớp CUDA của PyTorch, việc build DCNv3 op ở Section 3 gần như chắc chắn sẽ lỗi. Nhớ bật **Settings → Accelerator → GPU** trong Kaggle trước khi chạy.

In [2]:
!nvidia-smi
!nvcc -V || echo "⚠️  Không tìm thấy nvcc — nếu bước build DCNv3 (Section 3) lỗi, xem phần Troubleshooting ở cuối notebook."


Mon Jul  6 16:46:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             25W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Section 2 — Cài đặt dependencies

Theo đúng yêu cầu chính thức của `InternImage/detection`: PyTorch 1.11 + CUDA 11.3, `mmcv-full==1.5.0`, `mmdet==2.28.1`, `timm==0.6.11`. Kaggle image mặc định có sẵn PyTorch bản khác (thường mới hơn) nên cần cài đè.

Cell này chạy khá lâu (vài phút) vì phải build/tải mmcv-full.

In [ ]:


import os, glob
import sys
import json
from PIL import Image

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import cv2

PREDICT=True
VISUALIZE=False
sys.path.append("/kaggle/input/detection-wheel")

!mkdir /kaggle/working/packages
!cp -r /kaggle/input/pycocotools/* /kaggle/working/packages
os.chdir("/kaggle/working/packages/pycocotools-2.0.6/")
!python setup.py install -q
!pip install . --no-index --find-links /kaggle/working/packages/ -q

!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/addict-2.4.0-py3-none-any.whl
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/yapf-0.32.0-py2.py3-none-any.whl
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/terminal-0.4.0-py3-none-any.whl
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/terminaltables-3.1.10-py2.py3-none-any.whl

!pip install /kaggle/input/mmdet3-wheels/mmcv_full-1.7.1-cp310-cp310-linux_x86_64.whl
!cp -r /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/mmdetection/ /kaggle/working/
%cd /kaggle/working/mmdetection
!pip install -e . --no-deps
%cd /kaggle/working/

!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/mmdet-2.26.0-py3-none-any.whl
!pip install ensemble_boxes
!cp -r /kaggle/input/internimages-detection/ops_dcnv3 /kaggle/working/
%cd /kaggle/working/ops_dcnv3
!sh ./make.sh
%cd /kaggle/working/
!rm -rf mmdetection
!rm -rf packages
!rm -rf ops_dcnv3

/opt/conda/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/opt/conda/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/py

In [ ]:
# %%capture install_log
# # Gỡ torch/torchvision mặc định của Kaggle image để tránh xung đột version
# !pip uninstall -y torch torchvision torchaudio

# # PyTorch 1.11 + CUDA 11.3 (đúng version InternImage yêu cầu)
# !pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 \
#     -f https://download.pytorch.org/whl/torch_stable.html

# # mmcv-full + mmdet + timm đúng version
# !pip install -U openmim
# !mim install mmcv-full==1.5.0
# !pip install timm==0.6.11 mmdet==2.28.1

# # Các dependency phụ trợ
# !pip install opencv-python termcolor yacs pyyaml scipy
# !pip install "numpy==1.26.4" "pydantic==1.10.13" "yapf==0.40.1"
# !pip install huggingface_hub


In [ ]:
# In lại log cài đặt nếu cần debug (bị %%capture ẩn ở cell trên)
# install_log.show()

import torch, mmcv, mmdet
print("torch     :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("mmcv      :", mmcv.__version__)
print("mmdet     :", mmdet.__version__)
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))


## Section 3 — Clone InternImage & build custom CUDA op (DCNv3)

Đây là bước dễ lỗi nhất trên Kaggle vì cần `nvcc` khớp đúng CUDA 11.3 mà PyTorch vừa cài. Nếu `sh make.sh` lỗi, xem phần Troubleshooting cuối notebook (dùng bản `.whl` precompiled thay thế).

In [ ]:
import os
os.chdir(CFG["WORK_ROOT"])

if not os.path.exists(CFG["REPO_DIR"]):
    !git clone https://github.com/OpenGVLab/InternImage.git {CFG["REPO_DIR"]}
else:
    print("Repo đã tồn tại, bỏ qua clone.")

os.chdir(os.path.join(CFG["REPO_DIR"], "detection"))
print("cwd:", os.getcwd())


In [ ]:
# Build DCNv3 CUDA op
os.chdir(os.path.join(CFG["REPO_DIR"], "detection", "ops_dcnv3"))
!sh ./make.sh
!python test.py   # phải in ra "All checking is True" ở cuối nếu build thành công
os.chdir(os.path.join(CFG["REPO_DIR"], "detection"))


## Section 4 — Tải checkpoint pretrain COCO

Dùng `huggingface_hub` để tự tìm đúng tên file trong repo `OpenGVLab/InternImage` (tránh hardcode sai tên do có nhiều schedule 1x/3x cho từng backbone size), tải checkpoint **detector đầy đủ** (không chỉ backbone) — vì checkpoint này đã có backbone + FPN + RPN + bbox_head pretrain trên COCO, khớp thẳng vào model Faster R-CNN sau khi bỏ mask_head (phần mask_head trong checkpoint sẽ tự bị bỏ qua khi load, không gây lỗi).

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

repo_id = "OpenGVLab/InternImage"
files = list_repo_files(repo_id)

prefix = f"mask_rcnn_internimage_{CFG['BACKBONE_SIZE']}_fpn_{CFG['SCHEDULE']}_coco"
candidates = [f for f in files if f.startswith(prefix) and f.endswith(".pth")]

if not candidates:
    # fallback: bỏ điều kiện schedule, lấy bất kỳ bản nào của đúng backbone size
    prefix_loose = f"mask_rcnn_internimage_{CFG['BACKBONE_SIZE']}_fpn"
    candidates = [f for f in files if f.startswith(prefix_loose) and f.endswith(".pth")]

assert candidates, (
    f"Không tìm thấy checkpoint khớp '{prefix}*' trong {repo_id}. "
    f"Hãy vào https://huggingface.co/{repo_id}/tree/main để tìm tên file đúng và tải tay."
)

ckpt_filename = sorted(candidates)[0]
print("Checkpoint được chọn:", ckpt_filename)

ckpt_path = hf_hub_download(repo_id=repo_id, filename=ckpt_filename,
                             local_dir=CFG["PRETRAIN_DIR"])
CFG["PRETRAINED_CKPT"] = ckpt_path
print("Đã tải về:", ckpt_path)


## Section 5 — Kiểm tra dataset đã chia sẵn train/val/test

Dataset đầu vào **đã được chia sẵn** thành 3 split (train/val/test) từ trước, nên notebook này không chia lại. Bước này chỉ đọc 3 file COCO JSON để in thống kê nhanh (số ảnh, số instance) và thống kê kích thước bbox (để xác nhận scale nhỏ 4–18px) — con số quan trọng khi diễn giải `AP_small` sau này.

In [ ]:
import json
import numpy as np

def load_coco(path):
    with open(path) as f:
        return json.load(f)

train_coco = load_coco(CFG["TRAIN_JSON"])
val_coco   = load_coco(CFG["VAL_JSON"])
test_coco  = load_coco(CFG["TEST_JSON"])

categories = train_coco["categories"]

for name, coco in (("Train", train_coco), ("Val", val_coco), ("Test", test_coco)):
    print(f"{name:5s}: {len(coco['images'])} ảnh / {len(coco['annotations'])} instance")

# Thống kê kích thước bbox (cạnh ngắn hơn của w/h) trên toàn bộ annotation train,
# để xác nhận scale nhỏ
all_train_anns = train_coco["annotations"]
sizes = np.array([min(a["bbox"][2], a["bbox"][3]) for a in all_train_anns])
print("\nPhân bố cạnh nhỏ nhất của bbox trên tập train (px):")
for p in [0, 5, 25, 50, 75, 95, 100]:
    print(f"  p{p:>3}: {np.percentile(sizes, p):.1f}px")
n_small = (sizes < 32).mean() * 100
print(f"\n{n_small:.1f}% instance có cạnh < 32px -> rơi vào nhóm 'small' theo định nghĩa COCO (AP_small là chỉ số quan trọng nhất, không phải AP tổng).")


In [ ]:
# Kiểm tra nhanh: mọi ảnh trong từng JSON có tồn tại thật trên đĩa không
def check_missing(coco, images_dir):
    return [im["file_name"] for im in coco["images"]
            if not os.path.exists(os.path.join(images_dir, im["file_name"]))]

any_missing = False
for name, coco, images_dir in (
    ("Train", train_coco, CFG["TRAIN_IMAGES_DIR"]),
    ("Val", val_coco, CFG["VAL_IMAGES_DIR"]),
    ("Test", test_coco, CFG["TEST_IMAGES_DIR"]),
):
    missing = check_missing(coco, images_dir)
    if missing:
        any_missing = True
        print(f"⚠️  {name}: {len(missing)} ảnh được liệt kê trong JSON nhưng không tìm thấy file, ví dụ:")
        print(missing[:5])
    else:
        print(f"OK — {name}: toàn bộ ảnh trong JSON đều tồn tại trên đĩa.")

if not any_missing:
    print("\nOK — toàn bộ 3 split đều hợp lệ.")


## Section 6 — Custom hook để log epoch summary ra CSV

MMDetection đã tự log loss/lr/thời gian/memory mỗi `log_interval` iteration ra stdout + file `.log.json` (Section 9 sẽ parse file này để vẽ đồ thị). Hook dưới đây bổ sung thêm 1 dòng tóm tắt **mỗi epoch** (dễ theo dõi bằng mắt hơn khi train nhiều epoch), ghi ra `epoch_summary.csv`.

In [ ]:
custom_hook_code = '''
import csv, os, time
import torch
from mmcv.runner import HOOKS, Hook


@HOOKS.register_module()
class EpochSummaryCSVHook(Hook):
    # Ghi tom tat loss / lr / thoi gian / VRAM sau moi epoch train vao 1 file CSV.

    def __init__(self, csv_path):
        self.csv_path = csv_path
        self._epoch_start = None
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        if not os.path.exists(csv_path):
            with open(csv_path, "w", newline="") as f:
                csv.writer(f).writerow(
                    ["epoch", "avg_loss", "lr", "epoch_time_sec", "gpu_mem_MB"])

    def before_train_epoch(self, runner):
        self._epoch_start = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def after_train_epoch(self, runner):
        elapsed = time.time() - self._epoch_start if self._epoch_start else float("nan")
        # log_buffer.output chi duoc dien sau khi goi average(); neu khong goi truoc,
        # hook nay chay truoc TextLoggerHook (priority thap hon) se luon doc duoc dict rong -> nan.
        if not runner.log_buffer.output:
            runner.log_buffer.average()
        avg_loss = runner.log_buffer.output.get("loss", float("nan"))
        lr = runner.current_lr()
        lr = lr[0] if isinstance(lr, (list, tuple)) else lr
        mem = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0

        with open(self.csv_path, "a", newline="") as f:
            csv.writer(f).writerow(
                [runner.epoch + 1, f"{avg_loss:.4f}", f"{lr:.2e}",
                 f"{elapsed:.1f}", f"{mem:.0f}"])

        print(f"[Epoch {runner.epoch + 1}] avg_loss={avg_loss:.4f}  lr={lr:.2e}  "
              f"time={elapsed:.1f}s  gpu_mem={mem:.0f}MB")


@HOOKS.register_module()
class PosRoIStatsHook(Hook):
    # Theo doi so luong RoI duong/am (positive/negative) ma bbox_sampler chon duoc
    # moi iteration. Neu avg_pos_roi ~ 0 lien tuc -> assigner/anchor khong khop duoc
    # GT nao (giai thich vi sao loss_bbox=0.0000 va acc=100% nhung mo hinh khong hoc duoc gi).

    def __init__(self, interval=20):
        self.interval = interval
        self._pos_counts = []
        self._neg_counts = []

    def before_run(self, runner):
        model = runner.model.module if hasattr(runner.model, "module") else runner.model
        sampler = model.roi_head.bbox_sampler
        orig_sample = sampler.sample
        pos_counts, neg_counts = self._pos_counts, self._neg_counts

        def patched_sample(*args, **kwargs):
            result = orig_sample(*args, **kwargs)
            pos_counts.append(len(result.pos_inds))
            neg_counts.append(len(result.neg_inds))
            return result

        sampler.sample = patched_sample

    def after_train_iter(self, runner):
        if not self._pos_counts or (runner.iter + 1) % self.interval != 0:
            return
        pos = self._pos_counts[-self.interval:]
        neg = self._neg_counts[-self.interval:]
        avg_pos = sum(pos) / len(pos)
        avg_neg = sum(neg) / len(neg)
        # Chi canh bao khi avg_pos THUC SU thap, khong in co dinh 1 cau bat ke gia tri.
        verdict = (
            "avg_pos_roi thap bat thuong => assigner/anchor van chua khop GT tot"
            if avg_pos < 1.0 else "co positive RoI on dinh, khong con hien tuong 0 positive")
        # TextLoggerHook lam tron loss_bbox ve 4 chu so (%.4f) nen 0.0000 co the dang
        # che mot gia tri rat nho nhung khac 0. Doc gia tri chua lam tron truc tiep tu
        # runner.outputs["log_vars"] de biet chinh xac 0.0000 la that hay chi la lam tron.
        raw_loss_bbox = float("nan")
        if isinstance(getattr(runner, "outputs", None), dict):
            raw_loss_bbox = runner.outputs.get("log_vars", {}).get("loss_bbox", float("nan"))
        runner.logger.info(
            f"[PosRoIStats] iter {runner.iter + 1}: avg_pos_roi={avg_pos:.1f}  "
            f"avg_neg_roi={avg_neg:.1f}  raw_loss_bbox={raw_loss_bbox:.8f}  ({verdict})")
'''

hook_path = os.path.join(CFG["REPO_DIR"], "detection", "custom_hooks.py")
with open(hook_path, "w") as f:
    f.write(custom_hook_code)

# kiem tra syntax truoc khi dung trong config
import ast
ast.parse(custom_hook_code)
print("Đã viết custom hook tại:", hook_path)


## Section 7 — Viết config finetune

Kế thừa (`_base_`) trực tiếp config chính thức `mask_rcnn_internimage_{t|s|b|l|xl}_fpn_{1x|3x}_coco.py` từ repo — nhờ vậy giữ nguyên chính xác toàn bộ hyperparameter kiến trúc backbone (channels/depths/groups...) do OpenGVLab định nghĩa, không cần tự chép lại (dễ sai). Phần override chỉ tập trung vào:

1. Xóa `mask_head` / `mask_roi_extractor` → Faster R-CNN
2. Pipeline dữ liệu: `with_mask=False`, resize cố định 800×800 (ảnh đã đúng size)
3. `anchor_generator.scales` nhỏ hơn + nới ngưỡng IoU của assigner cho object nhỏ
4. Dataset custom (đường dẫn, 1 lớp `ship`)
5. Lịch train finetune (lr nhỏ, ít epoch hơn train from scratch)
6. `fp16` + `with_cp=True` để tiết kiệm VRAM
7. `custom_hooks` gọi `EpochSummaryCSVHook` viết ở Section 6

In [ ]:
base_config_rel = f"configs/coco/mask_rcnn_internimage_{CFG['BACKBONE_SIZE']}_fpn_{CFG['SCHEDULE']}_coco.py"
base_config_abs = os.path.join(CFG["REPO_DIR"], "detection", base_config_rel)
assert os.path.exists(base_config_abs), f"Không tìm thấy base config: {base_config_abs}"
print("Base config:", base_config_rel)


In [ ]:
finetune_config = '''
_base_ = ["./{base_config_rel_name}"]

custom_imports = dict(imports=["custom_hooks"], allow_failed_imports=False)

# ------------------------------------------------------------------
# 1) Kien truc: bo mask_head / mask_roi_extractor -> Faster R-CNN
#    (Mask R-CNN = Faster R-CNN + mask head, nen chi can xoa nhanh mask)
# ------------------------------------------------------------------
model = dict(
    backbone=dict(init_cfg=None, with_cp=True),  # bo init_cfg ImageNet vi load_from se nap toan bo detector
    rpn_head=dict(
        anchor_generator=dict(scales=[1, 2, 4])),  # mac dinh scales=[8] -> anchor nho nhat 32px, qua to cho tau 4-18px
    roi_head=dict(
        bbox_head=dict(num_classes={num_classes}),
        mask_roi_extractor=None,
        mask_head=None),
    train_cfg=dict(
        rpn=dict(
            # GT tau qua nho (4-18px) nen anchor thuong lech tam vai px la du de IoU
            # tut duoi 0.5-0.7 mac dinh; ha nguong de RPN co positive sample on dinh hon.
            assigner=dict(pos_iou_thr=0.3, neg_iou_thr=0.3, min_pos_iou=0.3)),
        rcnn=dict(
            # match_low_quality=True: neu khong proposal nao dat pos_iou_thr, van gan
            # cuong buc GT cho proposal co IoU cao nhat, tranh RCNN khong co positive RoI
            # nao (=> loss_bbox=0.0000, acc=100% gia, val rong nhu da quan sat duoc).
            assigner=dict(pos_iou_thr=0.3, neg_iou_thr=0.3, min_pos_iou=0.1,
                          match_low_quality=True),
            sampler=dict(add_gt_as_proposals=True))))

# ------------------------------------------------------------------
# 2) Dataset
# ------------------------------------------------------------------
dataset_type = "CocoDataset"
classes = {classes_tuple}
data_root = "{data_dir}/"

img_norm_cfg = dict(
    mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375], to_rgb=True)

train_pipeline = [
    dict(type="LoadImageFromFile"),
    dict(type="LoadAnnotations", with_bbox=True, with_mask=False),
    dict(type="Resize", img_scale=({img_size}, {img_size}), keep_ratio=True),
    dict(type="RandomFlip", flip_ratio=0.5),
    dict(type="PhotoMetricDistortion"),
    dict(type="Normalize", **img_norm_cfg),
    dict(type="Pad", size_divisor=32),
    dict(type="DefaultFormatBundle"),
    dict(type="Collect", keys=["img", "gt_bboxes", "gt_labels"]),
]
test_pipeline = [
    dict(type="LoadImageFromFile"),
    dict(
        type="MultiScaleFlipAug",
        img_scale=({img_size}, {img_size}),
        flip=False,
        transforms=[
            dict(type="Resize", keep_ratio=True),
            dict(type="RandomFlip"),
            dict(type="Normalize", **img_norm_cfg),
            dict(type="Pad", size_divisor=32),
            dict(type="ImageToTensor", keys=["img"]),
            dict(type="Collect", keys=["img"]),
        ])
]

data = dict(
    samples_per_gpu={batch_size},
    workers_per_gpu={workers},
    train=dict(
        _delete_=True,
        type=dataset_type,
        classes=classes,
        ann_file="{train_json}",
        img_prefix="{train_images_dir}/",
        pipeline=train_pipeline),
    val=dict(
        _delete_=True,
        type=dataset_type,
        classes=classes,
        ann_file="{val_json}",
        img_prefix="{val_images_dir}/",
        pipeline=test_pipeline),
    test=dict(
        _delete_=True,
        type=dataset_type,
        classes=classes,
        ann_file="{test_json}",
        img_prefix="{test_images_dir}/",
        pipeline=test_pipeline))

# ------------------------------------------------------------------
# 3) Lich finetune (lr nho + it epoch hon train-from-scratch)
# ------------------------------------------------------------------
optimizer = dict(lr={lr})
lr_config = dict(warmup_iters={warmup_iters}, step={lr_step})
runner = dict(type="EpochBasedRunner", max_epochs={max_epochs})

evaluation = dict(interval={eval_interval}, metric=["bbox"], save_best="bbox_mAP")
checkpoint_config = dict(interval=1, max_keep_ckpts={max_keep_ckpts})

log_config = dict(
    interval={log_interval},
    hooks=[
        dict(type="TextLoggerHook"),
    ])

custom_hooks = [
    dict(type="EpochSummaryCSVHook", csv_path="{work_dir}/epoch_summary.csv"),
    dict(type="PosRoIStatsHook", interval={log_interval}),
]

# ------------------------------------------------------------------
# 4) Toi uu VRAM tren GPU Kaggle (T4/P100 ~16GB)
# ------------------------------------------------------------------
fp16 = dict(loss_scale="dynamic")

load_from = "{pretrained_ckpt}"
seed = {seed}
gpu_ids = range(1)
work_dir = "{work_dir}"
'''.format(
    base_config_rel_name=os.path.basename(base_config_rel),
    num_classes=len(CFG["CLASSES"]),
    classes_tuple=tuple(CFG["CLASSES"]),
    data_dir=CFG["DATA_DIR"],
    img_size=CFG["IMG_SIZE"],
    batch_size=CFG["BATCH_SIZE"],
    workers=CFG["WORKERS"],
    train_json=CFG["TRAIN_JSON"],
    val_json=CFG["VAL_JSON"],
    test_json=CFG["TEST_JSON"],
    train_images_dir=CFG["TRAIN_IMAGES_DIR"],
    val_images_dir=CFG["VAL_IMAGES_DIR"],
    test_images_dir=CFG["TEST_IMAGES_DIR"],
    lr=CFG["LR"],
    warmup_iters=CFG["WARMUP_ITERS"],
    lr_step=CFG["LR_STEP"],
    max_epochs=CFG["MAX_EPOCHS"],
    eval_interval=CFG["EVAL_INTERVAL"],
    max_keep_ckpts=CFG["CHECKPOINT_MAX_KEEP"],
    log_interval=CFG["LOG_INTERVAL"],
    work_dir=CFG["WORK_DIR"],
    pretrained_ckpt=CFG["PRETRAINED_CKPT"],
    seed=CFG["SEED"],
)

finetune_config_path = os.path.join(CFG["REPO_DIR"], "detection", "configs", "coco",
                                     "finetune_ship_internimage.py")
with open(finetune_config_path, "w") as f:
    f.write(finetune_config)

CFG["FINETUNE_CONFIG"] = finetune_config_path
print("Đã viết config finetune tại:", finetune_config_path)
print("-" * 60)
print(finetune_config)


In [ ]:
# Sanity check: load thử config bằng mmcv.Config để phát hiện lỗi cú pháp/merge sớm,
# TRƯỚC KHI tốn thời gian chạy train.py
from mmcv import Config
cfg_check = Config.fromfile(CFG["FINETUNE_CONFIG"])
print("Config load OK.")
print("Số lớp (bbox_head.num_classes):", cfg_check.model.roi_head.bbox_head.num_classes)
print("mask_head:", cfg_check.model.roi_head.get("mask_head"))
print("anchor scales:", cfg_check.model.rpn_head.anchor_generator.scales)
print("load_from:", cfg_check.load_from)


In [ ]:
# So sánh kích thước GT bbox SAU KHI RESIZE với kích thước anchor thực tế đang dùng.
# Mục đích: nếu phần lớn bbox sau resize nhỏ hơn anchor nhỏ nhất -> RPN gần như
# không thể tạo proposal có IoU đủ cao với GT -> RCNN không có positive RoI ->
# loss_bbox=0.0000 và acc=100% (giả, vì mô hình chỉ đoán background) như log training
# đang cho thấy, KHÔNG PHẢI dấu hiệu học tốt/overfit.
import mmcv
import numpy as np

img_scale = (CFG["IMG_SIZE"], CFG["IMG_SIZE"])  # giống img_scale trong train/test pipeline
id_to_size = {im["id"]: (im["width"], im["height"]) for im in train_coco["images"]}

# Dùng đúng hàm mmcv.rescale_size mà Resize(keep_ratio=True) gọi bên trong,
# thay vì tự ước lượng lại tỉ lệ resize.
resized_min_sides = []
for ann in train_coco["annotations"]:
    w, h = id_to_size[ann["image_id"]]
    _, scale_factor = mmcv.rescale_size((w, h), img_scale, return_scale=True)
    bw, bh = ann["bbox"][2], ann["bbox"][3]
    resized_min_sides.append(min(bw, bh) * scale_factor)

resized_min_sides = np.array(resized_min_sides)
print("Phân bố cạnh nhỏ nhất của GT bbox SAU RESIZE (px, img_scale=%s):" % (img_scale,))
for p in [0, 5, 25, 50, 75, 95, 100]:
    print(f"  p{p:>3}: {np.percentile(resized_min_sides, p):.1f}px")

# Kích thước anchor thực tế, lấy trực tiếp từ config đã ghi ra (cfg_check ở cell trên),
# không đoán lại từ code.
anchor_gen = cfg_check.model.rpn_head.anchor_generator
strides = anchor_gen["strides"]
scales = anchor_gen["scales"]
anchor_sizes = sorted(s * a for s in strides for a in scales)
print("\nKích thước anchor (cạnh, px) trên các FPN level (stride x scale):", anchor_sizes)
print("Anchor nhỏ nhất:", min(anchor_sizes), "px")

frac_below_min_anchor = (resized_min_sides < min(anchor_sizes)).mean() * 100
print(f"\n{frac_below_min_anchor:.1f}% GT bbox (sau resize) NHỎ HƠN anchor nhỏ nhất "
      f"-> RPN khó/không thể sinh proposal có IoU cao với các box này.")
if frac_below_min_anchor > 20:
    print("=> Cân nhắc giảm strides/scales nhỏ hơn nữa hoặc tăng IMG_SIZE để bbox sau resize lớn hơn.")


## Section 8 — Train

Chạy qua `train.py` của repo (không dùng `mmdet.apis.train_detector` trực tiếp trong notebook để tránh xung đột multiprocessing/logging handler khi chạy lại nhiều lần trong cùng 1 kernel Jupyter). Output được `tee` ra file `train_stdout.log` để xem lại nếu mất kết nối notebook.

Mỗi **20 iteration** (`LOG_INTERVAL`) sẽ in ra: `loss`, các loss con (`loss_rpn_cls`, `loss_rpn_bbox`, `loss_cls`, `loss_bbox`), `acc` (độ chính xác phân loại của RoI head), `lr`, `time`/`data_time` mỗi iteration, và `memory` (VRAM, MB) — đây là bộ chỉ số tối thiểu nên theo dõi khi finetune detector. Mỗi **epoch** sẽ có thêm 1 dòng tóm tắt từ `EpochSummaryCSVHook`, và mỗi **1 epoch** sẽ có `bbox_mAP`, `bbox_mAP_50`, `bbox_mAP_75`, `bbox_mAP_s`, `bbox_mAP_m`, `bbox_mAP_l` trên val set.

In [ ]:
os.chdir(os.path.join(CFG["REPO_DIR"], "detection"))
train_stdout = os.path.join(CFG["WORK_DIR"], "train_stdout.log")

!python train.py {CFG["FINETUNE_CONFIG"]} \
    --work-dir {CFG["WORK_DIR"]} \
    --seed {CFG["SEED"]} \
    --deterministic \
    2>&1 | tee {train_stdout}


## Section 9 — Trực quan hóa log (loss curve + mAP curve)

MMDetection tự ghi mọi log ra file `{work_dir}/{timestamp}.log.json` (mỗi dòng là 1 JSON — dòng `mode="train"` cho loss theo iteration, dòng `mode="val"` cho mAP theo epoch). Cell dưới tự tìm file log mới nhất và vẽ 2 đồ thị.

In [ ]:
import glob, json as _json
import matplotlib.pyplot as plt

log_json_files = sorted(glob.glob(os.path.join(CFG["WORK_DIR"], "*.log.json")))
assert log_json_files, "Không tìm thấy file log.json — kiểm tra lại Section 8 đã chạy xong chưa."
log_json_path = log_json_files[-1]
print("Đang đọc:", log_json_path)

train_rows, val_rows = [], []
with open(log_json_path) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = _json.loads(line)
        except _json.JSONDecodeError:
            continue
        if rec.get("mode") == "train" and "loss" in rec:
            train_rows.append(rec)
        elif rec.get("mode") == "val":
            val_rows.append(rec)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_rows:
    iters = list(range(len(train_rows)))
    axes[0].plot(iters, [r["loss"] for r in train_rows], label="total loss")
    for key in ["loss_rpn_cls", "loss_rpn_bbox", "loss_cls", "loss_bbox"]:
        if key in train_rows[0]:
            axes[0].plot(iters, [r.get(key, None) for r in train_rows], label=key, alpha=0.6)
    axes[0].set_xlabel(f"log step (mỗi {CFG['LOG_INTERVAL']} iteration)")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Training loss")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
else:
    axes[0].set_title("Chưa có dữ liệu train loss")

if val_rows:
    epochs = [r.get("epoch", i + 1) for i, r in enumerate(val_rows)]
    for key in ["bbox_mAP", "bbox_mAP_50", "bbox_mAP_s"]:
        if key in val_rows[0]:
            axes[1].plot(epochs, [r.get(key, None) for r in val_rows], marker="o", label=key)
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("mAP")
    axes[1].set_title("Validation mAP (bbox_mAP_s = AP cho object nhỏ)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
else:
    axes[1].set_title("Chưa có dữ liệu eval — kiểm tra evaluation.interval trong config")

plt.tight_layout()
plt.savefig(os.path.join(CFG["WORK_DIR"], "training_curves.png"), dpi=120)
plt.show()


In [ ]:
# Xem nhanh epoch_summary.csv (do custom hook ghi)
import pandas as pd

csv_path = os.path.join(CFG["WORK_DIR"], "epoch_summary.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)
else:
    print("Chưa có epoch_summary.csv (train chưa xong epoch nào).")


## Section 10 — Đánh giá cuối cùng trên test set (đầy đủ COCO metrics)

Chạy `test.py` với checkpoint tốt nhất (`best_bbox_mAP_*.pth`, do `save_best="bbox_mAP"` ở Section 7 tự lưu) trên **test set** (giữ riêng, chưa từng được dùng để chọn checkpoint hay tune hyperparameter) để in đầy đủ bảng `AP / AP50 / AP75 / AP_small / AP_medium / AP_large`.

In [ ]:
best_ckpts = sorted(glob.glob(os.path.join(CFG["WORK_DIR"], "best_bbox_mAP_*.pth")))
if best_ckpts:
    best_ckpt = best_ckpts[-1]
else:
    # fallback: checkpoint cuoi cung
    all_ckpts = sorted(glob.glob(os.path.join(CFG["WORK_DIR"], "epoch_*.pth")))
    assert all_ckpts, "Không tìm thấy checkpoint nào trong work_dir."
    best_ckpt = all_ckpts[-1]

print("Dùng checkpoint:", best_ckpt)
CFG["BEST_CKPT"] = best_ckpt

os.chdir(os.path.join(CFG["REPO_DIR"], "detection"))
!python test.py {CFG["FINETUNE_CONFIG"]} {best_ckpt} --eval bbox


## Section 11 — Lưu checkpoint & thử inference nhanh trên vài ảnh test

Copy checkpoint tốt nhất + config ra ngoài `/kaggle/working` (ngoài `work_dir`, để dễ tìm khi Save Version), và vẽ thử prediction lên 4 ảnh test ngẫu nhiên để kiểm tra bằng mắt.

In [ ]:
import shutil

final_dir = "/kaggle/working/final_model"
os.makedirs(final_dir, exist_ok=True)
shutil.copy(CFG["BEST_CKPT"], os.path.join(final_dir, "model.pth"))
shutil.copy(CFG["FINETUNE_CONFIG"], os.path.join(final_dir, "config.py"))
print("Đã lưu model cuối cùng vào:", final_dir)


In [ ]:
from mmdet.apis import init_detector, inference_detector, show_result_pyplot
import random

model = init_detector(CFG["FINETUNE_CONFIG"], CFG["BEST_CKPT"], device="cuda:0")

with open(CFG["TEST_JSON"]) as f:
    test_coco = _json.load(f)
sample_imgs = random.sample(test_coco["images"], k=min(4, len(test_coco["images"])))

for im in sample_imgs:
    img_path = os.path.join(CFG["TEST_IMAGES_DIR"], im["file_name"])
    result = inference_detector(model, img_path)
    show_result_pyplot(model, img_path, result, score_thr=0.3)


## Troubleshooting — các lỗi hay gặp trên Kaggle

**1. `sh make.sh` (Section 3) lỗi khi build DCNv3**
- Nguyên nhân phổ biến nhất: `nvcc` không tồn tại hoặc version không khớp CUDA 11.3 của PyTorch vừa cài ở Section 2.
- Kiểm tra: `!nvcc -V` — nếu không có gì in ra, image Kaggle hiện tại không có CUDA toolkit đầy đủ (chỉ có driver).
- Cách xử lý nhanh nhất: tìm bản `.whl` DCNv3 precompiled từ trang release của InternImage (link `DCNv3-1.0-whl` trong README của repo) khớp đúng Python/torch/CUDA version rồi `pip install` trực tiếp file `.whl` đó, thay vì build từ source.

**2. `mim install mmcv-full==1.5.0` không tìm được wheel phù hợp**
- Chỉ định rõ CUDA + torch version trong URL index:
  `pip install mmcv-full==1.5.0 -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.11.0/index.html`

**3. CUDA out of memory**
- Giảm `BATCH_SIZE` xuống 1 trong CFG (Section 0), chạy lại từ Section 7.
- Đảm bảo `fp16` và `backbone.with_cp=True` đã có trong config (đã bật sẵn ở Section 7).

**4. Muốn train nhanh thử nghiệm (smoke test) trước khi chạy full**
- Sửa `MAX_EPOCHS = 1` và `EVAL_INTERVAL = 1` trong CFG, chạy hết pipeline 1 lần để chắc mọi thứ chạy được, rồi mới tăng lên epoch thật.

**5. Kaggle giới hạn thời gian session (9-12h)**
- Với InternImage-T + 2000 ảnh + batch 2, mỗi epoch thường vài phút trên T4 — 12 epoch nên xong trong 1 session. Nếu cần train lại, dùng `--resume-from {work_dir}/latest.pth` khi gọi `train.py`.

**6. `AssertionError` liên quan `mask_head`/`mask_roi_extractor` khi load config**
- Đảm bảo bản mmdet đang dùng đúng `2.28.1` (không phải mmdet 3.x) — cú pháp `mask_head=None` để "xóa" nhánh mask chỉ hoạt động đúng với cơ chế merge config của mmcv 1.x/mmdet 2.x như đã cài ở Section 2.
